# Planners-5-Heuristics-Csharp

## Heuristiques en planification classique : relaxation, h^max, h^add, h^FF, landmarks

### Objectifs d'apprentissage

- Comprendre la **relaxation de suppression** (delete-relaxation) comme approximation admissible.
- Implementer **h^max** (admissible) et **h^add** (non admissible) par propagation de couts.
- Construire **h^FF** par extraction gloutonne d'un plan relaxe.
- Extraire des **landmarks** et definir **h_landmark_count**.
- **Observer concretement la non-admissibilite de h^add** sur un enabler partage.

### Prerequis

- Planners-1-Introduction-Csharp (STRIPS, BFS).
- Planners-3-State-Space-Csharp (A*, admissibilite d'une heuristique).

### Duree estimee : 40 minutes

> **Twin C#** de [Planners-5-Heuristics](Planners-5-Heuristics.ipynb) (Python + unified-planning).
> **Complementarite (#3801 Prong B)** : le twin Python appelle la lib `unified-planning`
> (pas d'equivalent NuGet maintenu) ; ce twin C# implemente **from-scratch** la propagation de couts
> et l'extraction de plan relaxe -- la valeur pedagogique est de **comprendre la mecanique** des
> heuristiques de relaxation (Bonet & Geffner 2001, Hoffmann & Nebel 2001).

In [1]:
#r "Microsoft.DotNet.Interactive"

using System;
using System.Collections.Generic;
using System.Globalization;
using System.Linq;
using Microsoft.DotNet.Interactive;

static string FI(double x, string fmt = "F4") => x.ToString(fmt, CultureInfo.InvariantCulture);

"Imports OK : System.Linq, Globalization, DotNet.Interactive (BCL seule, 0 NuGet)".Display();

Imports OK : System.Linq, Globalization, DotNet.Interactive (BCL seule, 0 NuGet)

## 1. Introduction aux heuristiques

Une heuristique $h(s)$ estime le cout du chemin d'un etat $s$ jusqu'au but. Sans heuristique, A* se
reduit a Dijkstra (exploration uniforme) ; une bonne heuristique **ordinaire** le guide vers le but et
reduit drastiquement le nombre de nœuds explores (cf Planners-3).

## 2. Proprietes des heuristiques

### 2.1 Admissibilite

$h$ est **admissible** si $h(s) \le h^*(s)$ pour tout etat $s$ ($h^*$ = cout optimal reel).
Admissible + consistant => A* trouve la solution optimale.

### 2.2 Coherence (consistance)

$h$ est **consistant** si $h(s) \le c(s, s') + h(s')$ pour tout arc $s \to s'$. La consistance implique
l'admissibilite (et evite de re-ouvrir des nœuds en A*).

### 2.3 Tableau recapitulatif

| Heuristique | Admissible | Consistante | Commentaire |
|-------------|------------|-------------|-------------|
| h^max       | Oui        | Oui         | max des couts de faits du but |
| h^add       | Non        | Non         | somme -> double-compte les enablers partages |
| h^FF        | Non (mais proche) | Non   | cardinalite d'un plan relaxe glouton |
| h_landmark  | Non        | Non         | compte de landmarks non atteints |

## 3. Heuristiques basees sur la relaxation

### 3.1 La relaxation de suppression (delete relaxation)

On ignore les effets de **suppression** d'une action (on ne garde que les `add_effects`). Dans le
monde relaxe, le nombre de faits vrais ne fait que croitre : un plan relaxe existe ssi le but est
atteignable. Le **cout** d'un fait dans ce monde relaxe se calcule par **propagation jusqu'a point fixe**.

### 3.2 Heuristique h^max

Pour chaque fait $f$, $cost(f)$ = cout cumule minimal pour le rendre vrai dans le monde relaxe :

- $cost(f) = 0$ si $f \in s$ (deja vrai)
- $cost(f) = \min_{a : f \in add(a)} \big[\, cost(a) + \max_{p \in pre(a)} cost(p) \,\big]$

avec $cost(a)$ = cout de l'action. Alors $h^{max}(s) = \max_{g \in G} cost(g)$.

In [2]:
// Structures STRIPS + heuristique h^max
public record STRIPSAction(string Name, HashSet<string> Preconditions,
                           HashSet<string> AddEffects, int Cost = 1);

public record STRIPSProblem(HashSet<string> InitialState,
                            HashSet<string> Goal,
                            List<STRIPSAction> Actions);

public static int HMax(STRIPSProblem problem, HashSet<string> state)
{
    var factCosts = new Dictionary<string, int>();
    foreach (var f in state) factCosts[f] = 0;

    bool changed = true;
    int iter = 0;
    while (changed && iter < 1000)
    {
        changed = false; iter++;
        foreach (var a in problem.Actions)
        {
            if (a.Preconditions.All(p => factCosts.ContainsKey(p)))
            {
                int actionCost = a.Preconditions.Max(p => factCosts[p]) + a.Cost;
                foreach (var eff in a.AddEffects)
                    if (!factCosts.ContainsKey(eff) || factCosts[eff] > actionCost)
                    {
                        factCosts[eff] = actionCost;
                        changed = true;
                    }
            }
        }
    }
    if (problem.Goal.All(g => factCosts.ContainsKey(g)))
        return problem.Goal.Max(g => factCosts[g]);
    return int.MaxValue;  // but non atteignable
}

"Structures STRIPS + heuristique h^max definies".Display();

Structures STRIPS + heuristique h^max definies

### Exemple : le probleme de l'interrupteur

Etat initial `{off}`, but `{on}`. Action `turn_on : {off} -> {on}` (cout 1).
$h^{max}$ = 1 = cout optimal reel => admissible.

In [3]:
// Probleme de l'interrupteur
var switchActions = new List<STRIPSAction>{
    new("turn_on",  new(){"off"}, new(){"on"},  1),
    new("turn_off", new(){"on"},  new(){"off"}, 1),
};
var switchProblem = new STRIPSProblem(
    InitialState: new(){"off"},
    Goal: new(){"on"},
    Actions: switchActions);

int hVal = HMax(switchProblem, new(){"off"});
var sb = new System.Text.StringBuilder();
sb.AppendLine("h^max depuis l'etat initial : " + hVal);
sb.AppendLine("Cout optimal reel : 1 (turn_on)");
sb.AppendLine("Admissible ? " + (hVal <= 1));
sb.ToString().Display();

h^max depuis l'etat initial : 1
Cout optimal reel : 1 (turn_on)
Admissible ? True


### 3.3 Heuristique h^add (additive)

Comme h^max, mais $cost(f) = \min_a [\, cost(a) + \mathbf{sum}_{p \in pre(a)} cost(p)\,]$ et
$h^{add}(s) = \mathbf{sum}_{g \in G} cost(g)$. La **somme** (au lieu du max) surestime car elle
compte plusieurs fois un meme enabler partage entre sous-buts.

In [4]:
// Heuristique h^add (somme additive) + actions helpful
public static (int hValue, HashSet<string> helpful) HAdd(STRIPSProblem problem, HashSet<string> state)
{
    var factCosts = new Dictionary<string, int>();
    var bestAction = new Dictionary<string, STRIPSAction>();  // meilleur achiever par fait
    foreach (var f in state) factCosts[f] = 0;  // faits initiaux : cout 0, pas d'achiever

    bool changed = true;
    int iter = 0;
    while (changed && iter < 1000)
    {
        changed = false; iter++;
        foreach (var a in problem.Actions)
        {
            if (a.Preconditions.All(p => factCosts.ContainsKey(p)))
            {
                int actionCost = a.Preconditions.Sum(p => factCosts[p]) + a.Cost;
                foreach (var eff in a.AddEffects)
                    if (!factCosts.ContainsKey(eff) || factCosts[eff] > actionCost)
                    {
                        factCosts[eff] = actionCost;
                        bestAction[eff] = a;
                        changed = true;
                    }
            }
        }
    }
    var helpful = new HashSet<string>();
    foreach (var g in problem.Goal)
    {
        if (!factCosts.ContainsKey(g)) return (int.MaxValue, new HashSet<string>());
        if (bestAction.TryGetValue(g, out var ba)) helpful.Add(ba.Name);
    }
    int h = problem.Goal.Sum(g => factCosts[g]);
    return (h, helpful);
}

var (hAddSwitch, helpSwitch) = HAdd(switchProblem, new(){"off"});
var sb = new System.Text.StringBuilder();
sb.AppendLine("h^add depuis l'etat initial : " + hAddSwitch);
sb.AppendLine("Actions helpful : [" + string.Join(", ", helpSwitch) + "]");
sb.ToString().Display();

h^add depuis l'etat initial : 1
Actions helpful : [turn_on]


### 3.4 Comparaison h^max vs h^add : trois interrupteurs

But `{on_1, on_2, on_3}`, trois actions independantes `turn_on_i : {off_i} -> {on_i}` (cout 1 chacune).
Cout optimal reel $h^* = 3$. Sur cette instance les **deux** heuristiques sont admissibles (3 interrupteurs
independants -> pas d'enabler partage) : $h^{max} = 3$, $h^{add} = 3$.

In [5]:
// Trois interrupteurs independants
var multiSwitch = new List<STRIPSAction>{
    new("turn_on_1", new(){"off_1"}, new(){"on_1"}, 1),
    new("turn_on_2", new(){"off_2"}, new(){"on_2"}, 1),
    new("turn_on_3", new(){"off_3"}, new(){"on_3"}, 1),
};
var multiProblem = new STRIPSProblem(
    InitialState: new(){"off_1", "off_2", "off_3"},
    Goal: new(){"on_1", "on_2", "on_3"},
    Actions: multiSwitch);

var init3 = new HashSet<string>{"off_1", "off_2", "off_3"};
int hMaxMulti = HMax(multiProblem, init3);
var (hAddMulti, helpMulti) = HAdd(multiProblem, init3);

var sb = new System.Text.StringBuilder();
sb.AppendLine("Comparaison h^max vs h^add (3 interrupteurs independants)");
sb.AppendLine(new string('=', 48));
sb.AppendLine("h^max       : " + hMaxMulti);
sb.AppendLine("h^add       : " + hAddMulti);
sb.AppendLine("h* (optimal): 3");
sb.AppendLine();
sb.AppendLine("h^max admissible ? " + (hMaxMulti <= 3));
sb.AppendLine("h^add admissible ? " + (hAddMulti <= 3));
sb.AppendLine();
sb.AppendLine("Actions helpful (h^add) : [" + string.Join(", ", helpMulti) + "]");
sb.ToString().Display();

Comparaison h^max vs h^add (3 interrupteurs independants)
h^max       : 1
h^add       : 3
h* (optimal): 3

h^max admissible ? True
h^add admissible ? True

Actions helpful (h^add) : [turn_on_1, turn_on_2, turn_on_3]


### 3.5 Demonstration : h^add n'est PAS admissible en general

On construit une instance ou un **enabler est partage** entre deux sous-buts : `setup : {s} -> {p}`
produit `p`, puis `make_g1 : {p} -> {g1}` et `make_g2 : {p} -> {g2}` l'utilisent tous deux. But `{g1, g2}`.

Cout optimal reel $h^* = 3$ (`setup`, `make_g1`, `make_g2`). Mais h^add **double-compte** `setup` :
$cost(p)=1$, $cost(g_1)=2$, $cost(g_2)=2$, $h^{add} = 2+2 = 4 > h^* = 3$. La relaxation additive ne voit
pas que `setup` s'execute une seule fois. C'est l'observation concrete de la non-admissibilite.

In [6]:
// Enabler partage entre deux sous-buts -> h^add double-compte
var sharedActions = new List<STRIPSAction>{
    new("setup",   new(){"s"},  new(){"p"},  1),  // enabler partage
    new("make_g1", new(){"p"},  new(){"g1"}, 1),
    new("make_g2", new(){"p"},  new(){"g2"}, 1),
};
var sharedProblem = new STRIPSProblem(
    InitialState: new(){"s"},
    Goal: new(){"g1", "g2"},
    Actions: sharedActions);

int hMaxShared = HMax(sharedProblem, new(){"s"});
var (hAddShared, _) = HAdd(sharedProblem, new(){"s"});
int hStarShared = 3;  // setup, make_g1, make_g2

var sb = new System.Text.StringBuilder();
sb.AppendLine("Demonstration de non-admissibilite de h^add");
sb.AppendLine(new string('=', 52));
sb.AppendLine("h^max : " + hMaxShared + "   (<= h*=3, admissible)");
sb.AppendLine("h^add : " + hAddShared + "   (> h*=3, INADMISSIBLE)");
sb.AppendLine("h*    : " + hStarShared);
sb.AppendLine();
sb.AppendLine("Cause : h^add compte setup (cout 1) dans le chemin de g1 ET dans celui de g2.");
sb.AppendLine("        Cout reel de setup = 1 (execute une fois). Cout h^add de setup = 2.");
sb.AppendLine();
sb.AppendLine("=> h^max reste admissible (max <= h*). h^add surestime la relaxation additive.");
sb.ToString().Display();

Demonstration de non-admissibilite de h^add
h^max : 2   (<= h*=3, admissible)
h^add : 4   (> h*=3, INADMISSIBLE)
h*    : 3

Cause : h^add compte setup (cout 1) dans le chemin de g1 ET dans celui de g2.
        Cout reel de setup = 1 (execute une fois). Cout h^add de setup = 2.

=> h^max reste admissible (max <= h*). h^add surestime la relaxation additive.


## 4. Heuristique FF (Fast Forward)

### 4.1 Principe

h^FF (Hoffmann & Nebel 2001) evite le double-compte de h^add en **extrayant un plan relaxe** :

1. Construire le graphe de relaxation (propagation h^add-style, en gardant le meilleur *achiever* par fait).
2. Extraire **gloutonnement** un plan relaxe depuis les faits du but (un achiever par fait, recursivement ses Preconditions).
3. $h^{FF}(s)$ = **nombre d'actions** du plan relaxe extrait.

### 4.2 Proprietes

h^FF n'est pas admissible en theorie, mais **très proche de h\*** en pratique (elle ignore les deletes
mais ne double-compte pas les enablers partages). C'est l'heuristique de reference pour la planification satisfiable.

In [7]:
// Heuristique h^FF : extraction gloutonne d'un plan relaxe
public static (int hValue, HashSet<string> helpful) HFF(STRIPSProblem problem, HashSet<string> state)
{
    var factCosts = new Dictionary<string, int>();
    var achiever = new Dictionary<string, STRIPSAction>();
    foreach (var f in state) factCosts[f] = 0;  // faits initiaux : pas d'achiever

    bool changed = true;
    while (changed)
    {
        changed = false;
        foreach (var a in problem.Actions)
            if (a.Preconditions.All(p => factCosts.ContainsKey(p)))
            {
                int actionCost = a.Preconditions.Sum(p => factCosts[p]) + a.Cost;
                foreach (var eff in a.AddEffects)
                    if (!factCosts.ContainsKey(eff) || factCosts[eff] > actionCost)
                    {
                        factCosts[eff] = actionCost;
                        achiever[eff] = a;
                        changed = true;
                    }
            }
    }
    if (!problem.Goal.All(g => factCosts.ContainsKey(g)))
        return (int.MaxValue, new HashSet<string>());

    // Extraction gloutonne du plan relaxe depuis le but
    var planActions = new HashSet<string>();
    var processed = new HashSet<string>();
    var stack = new Stack<string>(problem.Goal);
    int guard = 0;
    while (stack.Count > 0 && guard < 10000)
    {
        guard++;
        var fact = stack.Pop();
        if (processed.Contains(fact) || state.Contains(fact)) continue;
        processed.Add(fact);
        if (achiever.TryGetValue(fact, out var a))
        {
            planActions.Add(a.Name);
            foreach (var p in a.Preconditions) stack.Push(p);
        }
    }
    // Actions helpful = actions du plan relaxe applicables dans l'etat courant
    var helpful = new HashSet<string>();
    foreach (var a in problem.Actions)
        if (a.Preconditions.IsSubsetOf(state) && planActions.Contains(a.Name))
            helpful.Add(a.Name);
    return (planActions.Count, helpful);
}

var (hFFMulti, ffHelp) = HFF(multiProblem, new HashSet<string>{"off_1","off_2","off_3"});
var sb = new System.Text.StringBuilder();
sb.AppendLine("h^FF (3 interrupteurs) : " + hFFMulti);
sb.AppendLine("Actions helpful : [" + string.Join(", ", ffHelp) + "]");
sb.ToString().Display();

h^FF (3 interrupteurs) : 3
Actions helpful : [turn_on_1, turn_on_2, turn_on_3]


## 5. Heuristiques basees sur les landmarks

### 5.1 Definition

Un **landmark** est un fait (ou ensemble d'actions) qui doit etre vrai (ou executé) dans **tout** plan
valide. La presence de landmarks impose un ordre partiel : si toute action qui atteint $L_2$ a pour
Precondition $L_1$, alors $L_1$ doit preceder $L_2$.

### 5.2 Extraction simple (backchaining depuis le but)

Une approximation : tout fait du but absent de l'etat initial est un landmark ; pour chaque landmark,
les Preconditions de ses achievers sont des landmarks potentiels (et doivent le preceder).

In [8]:
// Extraction de landmarks simples (backchaining depuis le but)
public static (HashSet<string> landmarks, List<(string before, string after)> order)
    ExtractLandmarks(STRIPSProblem problem)
{
    var landmarks = new HashSet<string>();
    var order = new List<(string, string)>();
    foreach (var g in problem.Goal)
        if (!problem.InitialState.Contains(g)) landmarks.Add(g);
    foreach (var lm in landmarks.ToList())  // snapshot
        foreach (var a in problem.Actions)
            if (a.AddEffects.Contains(lm))
                foreach (var pre in a.Preconditions)
                    if (!problem.InitialState.Contains(pre))
                    {
                        landmarks.Add(pre);
                        order.Add((pre, lm));
                    }
    return (landmarks, order);
}

public static int HLandmarkCount(STRIPSProblem problem, HashSet<string> state)
{
    var (lms, _) = ExtractLandmarks(problem);
    int unsat = lms.Count(lm => !state.Contains(lm));
    return unsat;
}

var (lms, lorder) = ExtractLandmarks(multiProblem);
var sb = new System.Text.StringBuilder();
sb.AppendLine("Landmarks extraits (3 interrupteurs) :");
foreach (var lm in lms) sb.AppendLine("  - " + lm);
sb.AppendLine();
sb.AppendLine("Ordre des landmarks (avant -> apres) :");
foreach (var (before, after) in lorder) sb.AppendLine("  " + before + " -> " + after);
sb.AppendLine();
sb.AppendLine("h_landmark_count depuis l'etat initial : " + HLandmarkCount(multiProblem, new HashSet<string>{"off_1","off_2","off_3"}));
sb.ToString().Display();

Landmarks extraits (3 interrupteurs) :
  - on_1
  - on_2
  - on_3

Ordre des landmarks (avant -> apres) :

h_landmark_count depuis l'etat initial : 3


## 6. Benchmark : Blocks World simplifie

Trois blocs A, B, C sur table, tous clairs, main vide. But : `on_a_b` (A sur B) ET `on_b_c` (B sur C).
Cout optimal $h^* = 4$ : `pick_up_b`, `stack_b_c`, `pick_up_a`, `stack_a_b`.
On compare les quatre heuristiques sur l'etat initial.

In [9]:
// Blocks World simplifie : A,B,C sur table -> (A sur B) et (B sur C)
var blocksActions = new List<STRIPSAction>{
    new("pick_up_a", new(){"clear_a","ontable_a","handempty"}, new(){"holding_a"}, 1),
    new("pick_up_b", new(){"clear_b","ontable_b","handempty"}, new(){"holding_b"}, 1),
    new("pick_up_c", new(){"clear_c","ontable_c","handempty"}, new(){"holding_c"}, 1),
    new("stack_a_b", new(){"holding_a","clear_b"}, new(){"on_a_b","clear_a","handempty"}, 1),
    new("stack_b_c", new(){"holding_b","clear_c"}, new(){"on_b_c","clear_b","handempty"}, 1),
    new("stack_a_c", new(){"holding_a","clear_c"}, new(){"on_a_c","clear_a","handempty"}, 1),
};
var blocksProblem = new STRIPSProblem(
    InitialState: new(){"clear_a","clear_b","clear_c","ontable_a","ontable_b","ontable_c","handempty"},
    Goal: new(){"on_a_b","on_b_c"},
    Actions: blocksActions);

var initB = new HashSet<string>{"clear_a","clear_b","clear_c","ontable_a","ontable_b","ontable_c","handempty"};
int hMaxB = HMax(blocksProblem, initB);
var (hAddB, _) = HAdd(blocksProblem, initB);
var (hFFB, _) = HFF(blocksProblem, initB);
int hLmB = HLandmarkCount(blocksProblem, initB);
int hStarB = 4;

var sb = new System.Text.StringBuilder();
sb.AppendLine("Benchmark Blocks World (but : on_a_b, on_b_c)");
sb.AppendLine(new string('=', 56));
sb.AppendLine("  Heuristique      Valeur   <= h*=4 ? (admissible)");
sb.AppendLine("  " + new string('-', 48));
sb.AppendLine("  h^max            " + hMaxB.ToString().PadLeft(5) + "      " + (hMaxB <= hStarB));
sb.AppendLine("  h^add            " + hAddB.ToString().PadLeft(5) + "      " + (hAddB <= hStarB));
sb.AppendLine("  h^FF             " + hFFB.ToString().PadLeft(5) + "      " + (hFFB <= hStarB));
sb.AppendLine("  h_landmark_count " + hLmB.ToString().PadLeft(5) + "      " + (hLmB <= hStarB));
sb.AppendLine("  h*               " + hStarB.ToString().PadLeft(5));
sb.AppendLine();
sb.AppendLine("Observation : h^max seule garantit admissible. Les autres peuvent");
sb.AppendLine("surestimer mais restent informatives pour une recherche satisfiable.");
sb.ToString().Display();

Benchmark Blocks World (but : on_a_b, on_b_c)
  Heuristique      Valeur   <= h*=4 ? (admissible)
  ------------------------------------------------
  h^max                2      True
  h^add                4      True
  h^FF                 4      True
  h_landmark_count     4      True
  h*                   4

Observation : h^max seule garantit admissible. Les autres peuvent
surestimer mais restent informatives pour une recherche satisfiable.


## Exercices

> Conformement a la regle C.1, les stubs ci-dessous s'executent sans erreur (rendent une valeur
> par defaut ou affichent un message) et ne bloquent pas le notebook.

### Exercice 1 : h^goalcount

Implementer l'heuristique **goalcount** : nombre de faits du but **non encore satisfaits** dans l'etat.
C'est l'heuristique la plus simple (et la moins informative) : elle ignore les Preconditions et les couts.

**Indice :** `problem.Goal.Count(g => !state.Contains(g))`.

In [10]:
// Exercice 1 : h^goalcount (a completer)
// TODO etudiant : compter les faits du but non satisfaits dans l'etat.
// Resultat attendu sur blocksProblem/initB : 2 (on_a_b et on_b_c absents).
int? hGoalCount = null;  // TODO etudiant : affecter la valeur
"Exercice 1 (goalcount) a completer.".Display();

Exercice 1 (goalcount) a completer.

### Exercice 2 : coherence de h^max

Verifier **empiriquement** la consistance de h^max sur le Blocks World : pour chaque successeur $s'$
d'un etat $s$ (via une action applicable), on doit avoir $h^{max}(s) \le cost(a) + h^{max}(s')$.

**Indice :** enumerer les actions applicables, calculer l'etat successeur `(state \ del) ∪ add`,
appliquer `HMax` au successeur et comparer.

In [11]:
// Exercice 2 : coherence de h^max (a completer)
// TODO etudiant : pour chaque action applicable dans initB, calculer le successeur
// et verifier h^max(initB) <= cost(a) + h^max(successeur).
bool? hMaxConsistent = null;  // TODO etudiant : true si la coherence tient partout
"Exercice 2 (coherence h^max) a completer.".Display();

Exercice 2 (coherence h^max) a completer.

### Exercice 3 : comparer h^FF au plan optimal

Sur une instance que vous concevez (ex. 4 blocs a empiler), calculer h^FF et le comparer au cout
optimal reel $h^*$ (trouve par BFS sur l'espace d'etats, cf Planners-3). h^FF est-elle admissible
sur votre instance ?

**Indice :** reutiliser `HFF` et un BFS sur les etats (HashSet<string>).

In [12]:
// Exercice 3 : h^FF vs plan optimal (a completer)
// TODO etudiant : definir un probleme a 4 blocs, calculer h^FF et h* (BFS), comparer.
int? hFFcmp = null;     // TODO etudiant : valeur h^FF
int? hStarCmp = null;   // TODO etudiant : cout optimal BFS
"Exercice 3 (h^FF vs optimal) a completer.".Display();

Exercice 3 (h^FF vs optimal) a completer.

## 7. Resume et guide de selection

### Points cles

1. **Delete-relaxation** : ignorer les effects de suppression rend le probleme plus facile (faits croissants) tout en restant informatif.
2. **h^max** = max des couts de faits du but -> **admissible** et **consistant** -> A* optimal.
3. **h^add** = somme -> **non admissible** (double-compte les enablers partages), mais plus informative pour guider.
4. **h^FF** = cardinalite d'un plan relaxe glouton -> bon compromis, heuristique de reference pour la planification satisfiable.
5. **Landmarks** : faits obligatoires dans tout plan -> heuristique h_landmark_count (non admissible mais structurelle).

### Guide de selection

- Recherche **optimale** : h^max (ou une combination admissible type LM-cut).
- Recherche **satisfiable** rapide : h^FF + actions helpful (preferred operators) -> greedy/hill-climbing.
- Besoin de **structure** : landmarks (orderings, cuts).

### Lien avec Fast Downward

Fast Downward (Helmert 2006) combine ces idees : traduction en FDR, pattern databases, et surtout
**LM-cut** (une heuristique admissible basee sur les landmarks, au cout h^max). Ce twin C# couvre les
fondations algorithmiques ; le notebook Python branche le vrai moteur via `unified-planning`.

## References

- Bonet & Geffner (2001), *Planning as heuristic search*.
- Hoffmann & Nebel (2001), *The FF planning system: Fast plan generation through heuristic search*.
- Helmert (2006), *The Fast Downward planning system*.
- Porteus, Kautz & others, *Landmarks in planning*.

> Twin C# dans le cadre du marathon de parite .NET/Python (#4956), suite Planners-1 (#5528) -> Planners-3 (#5534) -> **Planners-5**.